In [7]:
from astroquery.mpc import MPC
from astropy.time import Time
import astropy.units as u
import json

In [ ]:
# Function to read the JSON settings file
def get_config():
  
  # Get settings' directory
  settings_dir = r"../config/settings.json"

  # Open JSON file
  with open(settings_dir, "r") as file:
      settings = json.load(file)
  return settings

In [ ]:
# Functions to standardize input data

# Helper function to handle asteroid type (VERIFICAR CON FUENTE CONFIABLE LOS TIPOS Y PARÁMETROS DE CADA TIPO)
# Possible fields to take into account: neo, semimajor_axis, perihelion_distance, eccentricity, tisserand_jupiter
def _get_type_args(fields):
  args = {}
  match fields['type']:
    case "NEO":
      args['neo'] = 'is_not_null'
    case "MBA":
      args['semimajor_axis_min'] = 2.0
      args['semimajor_axis_max'] = 3.2
      args['perihelion_distance_min'] = 1.3
    case "Jupiter Trojan":
      args['semimajor_axis_min'] = 5.05
      args['semimajor_axis_max'] = 5.40
    case "Centaur":
      args['semimajor_axis_min'] = 5.2
      args['semimajor_axis_max'] = 30.1
    case "TNO":
      args['semimajor_axis_min'] = 30.1
      args['perihelion_distance_min'] = 30.1
  return args

# Helper function to handle provisional or normal state
def _get_state_args(fields):
  args = {}
  match fields['include']:
    case 'named':
      args['name'] = 'is_not_null'
      args['return_fields'] = 'name'
    case 'numbered':
      args['number'] = 'is_not_null'
      args['return_fields'] = 'number'
    case 'provisional':
      args['designation'] = 'is_not_null'
      args['return_fields'] = 'designation'
    case _:
      args['return_fields'] = 'name'
  return args

# Helper function to search body names
def _search_bodies(fields):
  direct_args = {
    "object_type": fields['object_type'] or 'asteroid',
    "orbit_uncertainty": fields['orbit_uncertainty'],
    "critical_list_numbered_object": fields["critical_list_numbered_object"],
    "limit": fields['limit'] or 300
  }
  state_args = _get_state_args(fields)
  type_args = _get_type_args(fields)
  
  args = direct_args | state_args | type_args
  
  args = {key: value for key, value in args.items() if value is not None}
  
  # MPC.query_objects(**args)
  
  return args

# Function to standardize input data
def default(settings):
  
  # Default values
  if not settings['limit_magnitude']:
    settings['limit_magnitude'] = 16
  if not settings['exposition_time']:
    settings['exposition_time'] = 5
  if not settings['database']:
    settings['database'] = ["JPL", "MPC"]
  if not settings['observer']['code'] and not settings['observer']['coord']:
    settings['observer']['code'] = "geocentric"
  if not settings['epoch']:
    settings['epoch'] = {
      "range": {
        "start": str(Time.now()),
        "stop": str(Time.now() + 0.5 * u.day),
        "step": "1m"
      }
    }
  if not settings['body']['id']:
    # settings['body'] = _search_bodies(settings['body']['fields'])
    args = _search_bodies(settings['body']['fields'])
    for i in args:
      print(f'{i}: {args[i]}')
    
    
  return settings

settings = get_config()
settings = default(settings)
# print(json.dumps(settings, indent=2))

object_type: asteroid
limit: 10
return_fields: name
